# **Пояснение для задачи «Магические руны Элдории»**

В этой задаче нужно определить, какие 5-символьные последовательности из символов `F`, `W` и `E` являются валидными заклинаниями.

На первый взгляд задача выглядит очень простой:

- алфавит маленький;
- длина строки фиксирована;
- объектов немного.

Но именно такие задачи часто оказываются нетривиальными:  
модель должна не просто “видеть буквы”, а уловить скрытые закономерности между их позициями и сочетаниями.


## **Почему эта задача вызвала споры**

Во время разбора задачи у участников возникли два лагеря:

- одни использовали бустинги и получали примерно 95–97 баллов;
- другие писали, что SVM даёт 100 баллов.

Это хороший пример того, что не существует “всемогущей” модели, которая всегда лучше остальных.

Для маленькой дискретной задачи с короткими строками и хорошо сконструированными признаками метод опорных векторов может оказаться сильнее популярных ансамблей деревьев.

В этом ноутбуке разобраны два подхода:

1. ансамбль моделей, который даёт сильный результат и позволяет анализировать неуверенные предсказания;
2. SVM с подбором гиперпараметров, который способен получить 100 баллов автоматически.

## **Общая идея признаков**

Строка длины 5 сама по себе неудобна для классических моделей машинного обучения, поэтому её нужно превратить в числовое описание.

Здесь используются три группы признаков:

1. **Позиционные one-hot признаки**  
   Для каждой позиции учитывается, какой символ в ней стоит.

2. **Попарные взаимодействия между позициями**  
   Важны не только отдельные символы, но и их комбинации в разных местах строки.

3. **Счётчики символов**  
   Сколько раз в строке встретились `F`, `W` и `E`.

Такое представление уже позволяет классическим моделям видеть не просто строку, а её структурные свойства.

In [1]:
import os
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [ ]:
TRAIN_PATH = "Data/train_runes.csv"
TEST_PATH = "Data/test_runes.csv"
OUTPUT_PATH = "Data/answers_you.csv"
REFERENCE_PATH = "Data/answers_true.csv"

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

train_df.head()

Train shape: (170, 2)
Test shape : (73, 1)


,rune,spell
0,WWWWF,1
1,FWEFW,1
2,FFEFF,0
3,EWFWE,1
4,EEEWF,0


## Подготовка признаков

Ниже определены вспомогательные функции, которые превращают строки с рунами в числовое представление.

Именно этот блок будет общим для обоих подходов.

In [4]:
def split_runes(series: pd.Series) -> pd.DataFrame:
    arr = series.astype(str).apply(list).tolist()
    return pd.DataFrame(arr, columns=[f"pos_{i}" for i in range(5)])


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_count_features(pos_df: pd.DataFrame) -> np.ndarray:
    count_w = (pos_df == "W").sum(axis=1).to_numpy().reshape(-1, 1)
    count_f = (pos_df == "F").sum(axis=1).to_numpy().reshape(-1, 1)
    count_e = (pos_df == "E").sum(axis=1).to_numpy().reshape(-1, 1)
    return np.hstack([count_w, count_f, count_e]).astype(np.float32)


def build_pairwise_interactions(X_ohe: np.ndarray) -> np.ndarray:
    groups = [
        [0, 1, 2],      # pos_0
        [3, 4, 5],      # pos_1
        [6, 7, 8],      # pos_2
        [9, 10, 11],    # pos_3
        [12, 13, 14],   # pos_4
    ]

    features = []

    for i in range(5):
        for j in range(i + 1, 5):
            for c1 in groups[i]:
                for c2 in groups[j]:
                    col = (X_ohe[:, c1] * X_ohe[:, c2]).reshape(-1, 1)
                    features.append(col)

    return np.hstack(features).astype(np.float32)


def build_features_fit(runes: pd.Series):
    pos_df = split_runes(runes)

    encoder = make_ohe()
    X_ohe = encoder.fit_transform(pos_df).astype(np.float32)

    X_pair = build_pairwise_interactions(X_ohe)
    X_count = build_count_features(pos_df)

    X = np.hstack([X_ohe, X_pair, X_count]).astype(np.float32)
    return X, encoder


def build_features_transform(runes: pd.Series, encoder):
    pos_df = split_runes(runes)

    X_ohe = encoder.transform(pos_df).astype(np.float32)
    X_pair = build_pairwise_interactions(X_ohe)
    X_count = build_count_features(pos_df)

    X = np.hstack([X_ohe, X_pair, X_count]).astype(np.float32)
    return X


def evaluate_with_reference(pred_df: pd.DataFrame, reference_path: str):
    if not os.path.exists(reference_path):
        print(f"\nЭталонный файл не найден: {reference_path}")
        return

    ref_df = pd.read_csv(reference_path)

    if not {"rune", "spell"}.issubset(ref_df.columns):
        print("\nВ эталонном файле должны быть колонки: rune, spell")
        return

    merged = pred_df.merge(ref_df, on="rune", suffixes=("_pred", "_true"))

    if len(merged) != len(pred_df):
        print("\nПредупреждение: не все руны совпали между prediction и reference.")

    y_pred = merged["spell_pred"].astype(int).to_numpy()
    y_true = merged["spell_true"].astype(int).to_numpy()

    hits = int((y_pred == y_true).sum())
    total = len(merged)
    acc = accuracy_score(y_true, y_pred)

    print("\n=== Автопроверка по эталону ===")
    print(f"Попаданий: {hits}/{total}")
    print(f"Accuracy : {acc:.6f}")

    errors = merged[merged["spell_pred"] != merged["spell_true"]].copy()
    if len(errors) == 0:
        print("Ошибок нет.")
    else:
        print("\nОшибочные руны:")
        print(errors[["rune", "spell_pred", "spell_true"]].to_string(index=False))

In [5]:
X_train, encoder = build_features_fit(train_df["rune"])
X_test = build_features_transform(test_df["rune"], encoder)
y = train_df["spell"].astype(int).to_numpy()

print("Размер train:", X_train.shape)
print("Размер test :", X_test.shape)

Размер train: (170, 108)
Размер test : (73, 108)


## Подход 1. Ансамбль моделей

Первый вариант - использовать несколько разных моделей и объединить их ответы.

*(для тех у кого возник вопрос: "Зачем что-то кроме бустинга?", внесу уточнение, что вариант с коробочным бустингом также просматривался и набрал 95 баллов в качестве ответа, а причиной того, что я отказался от включения его в разбор задания является то, что на примере ансамбля, в котором также есть бустинг - ситуация выглядит интереснее и лучше описывает главную мысль задачи)*

В этом подходе участвуют:

- `GradientBoostingClassifier`
- `RandomForestClassifier`
- `SVM`
- мягкое голосование (`soft voting`) по вероятностям

Идея в том, что разные модели могут ошибаться на разных объектах, а ансамбль способен сгладить часть ошибок.

Однако именно на этой задаче даже сильный ансамбль может застревать около 95-97 баллов.

In [6]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        probability=True,
        random_state=42
    ))
])

ensemble = VotingClassifier(
    estimators=[
        ("gb", gb),
        ("rf", rf),
        ("svm", svm)
    ],
    voting="soft",
    weights=[2, 1, 1],
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Почему здесь полезна кросс-валидация

В этой задаче данных немного, поэтому одной случайной разбивки может быть недостаточно.

`StratifiedKFold` позволяет:

- сохранить баланс классов в каждом фолде;
- увидеть, насколько стабильно ведёт себя модель;
- сравнить разные алгоритмы более честно.

После этого можно обучить модели на всём train и посмотреть, как они ведут себя на test.

In [7]:
gb.fit(X_train, y)
rf.fit(X_train, y)
svm.fit(X_train, y)
ensemble.fit(X_train, y)

p_gb = gb.predict_proba(X_test)[:, 1]
p_rf = rf.predict_proba(X_test)[:, 1]
p_svm = svm.predict_proba(X_test)[:, 1]
p_ens = ensemble.predict_proba(X_test)[:, 1]

test_pred_ens = (p_ens >= 0.5).astype(int)

submission_ens = pd.DataFrame({
    "rune": test_df["rune"],
    "spell": test_pred_ens
})

submission_ens.to_csv("answers_ensemble.csv", index=False)

print("Файл сохранён: answers_ensemble.csv")
submission_ens.head(10)

Файл сохранён: answers_ensemble.csv


,rune,spell
0,FWFEF,1
1,EFEEE,0
2,EFWFE,0
3,FEWWF,0
4,WFEFE,0
5,EWFEW,0
6,WWFFW,1
7,FEWFW,0
8,WEFEW,1
9,EFFFF,0


In [8]:
uncertainty_df = pd.DataFrame({
    "rune": test_df["rune"],
    "p_gb": p_gb,
    "p_rf": p_rf,
    "p_svm": p_svm,
    "p_ens": p_ens,
    "pred": test_pred_ens,
    "uncertainty": np.abs(p_ens - 0.5)
}).sort_values("uncertainty", ascending=True)

print("Топ-10 самых неуверенных предсказаний ансамбля:")
print(uncertainty_df.head(10).to_string(index=False))

Топ-10 самых неуверенных предсказаний ансамбля:
 rune     p_gb   p_rf    p_svm    p_ens  pred  uncertainty
WFWFW 0.487228 0.3300 0.404085 0.427135     0     0.072865
FFWFW 0.248875 0.3675 0.170830 0.259020     0     0.240980
WFWEW 0.979132 0.6475 0.981572 0.896834     1     0.396834
FFWEW 0.977958 0.6925 0.949681 0.899524     1     0.399524
EWWWW 0.982039 0.6850 0.953895 0.900743     1     0.400743
WWWEE 0.018317 0.3125 0.031193 0.095082     0     0.404918
FEWWF 0.008670 0.3300 0.010649 0.089497     0     0.410503
EWWWF 0.988561 0.6975 0.987799 0.915605     1     0.415605
FWWEF 0.022069 0.2800 0.005615 0.082438     0     0.417562
EFWWE 0.005159 0.2375 0.061785 0.077401     0     0.422599


In [9]:
disagreement_df = pd.DataFrame({
    "rune": test_df["rune"],
    "p_gb": p_gb,
    "p_rf": p_rf,
    "p_svm": p_svm,
    "p_ens": p_ens,
    "pred": test_pred_ens
})

disagreement_df["disagreement"] = (
    disagreement_df[["p_gb", "p_rf", "p_svm"]].max(axis=1)
    - disagreement_df[["p_gb", "p_rf", "p_svm"]].min(axis=1)
)

disagreement_df = disagreement_df.sort_values("disagreement", ascending=False)

print("Топ-10 объектов с максимальным расхождением моделей:")
print(disagreement_df.head(10).to_string(index=False))

Топ-10 объектов с максимальным расхождением моделей:
 rune     p_gb   p_rf    p_svm    p_ens  pred  disagreement
WFWEW 0.979132 0.6475 0.981572 0.896834     1      0.334072
FEWWF 0.008670 0.3300 0.010649 0.089497     0      0.321330
EWWWW 0.982039 0.6850 0.953895 0.900743     1      0.297039
WWWEE 0.018317 0.3125 0.031193 0.095082     0      0.294183
FEEFW 0.994508 0.7050 0.996752 0.922692     1      0.291752
EWWWF 0.988561 0.6975 0.987799 0.915605     1      0.291061
FFWEW 0.977958 0.6925 0.949681 0.899524     1      0.285458
FWWEF 0.022069 0.2800 0.005615 0.082438     0      0.274385
FFFFW 0.988334 0.7375 0.995487 0.927414     1      0.257987
WEWWW 0.006031 0.2625 0.009129 0.070923     0      0.256469


## Что показывает анализ неуверенности

Даже если ансамбль не даёт 100 баллов автоматически, он всё равно полезен.

Он позволяет понять:

- какие объекты находятся близко к границе решения;
- где разные модели особенно сильно спорят;
- какие последовательности являются самыми “сомнительными”.

В пост-контестном анализе это может быть особенно интересно:  
если уже известно, что ошибка всего одна, можно изучить наиболее неуверенные предсказания и понять, какая именно строка ломает результат.

Именно так можно объяснить, почему участники иногда вручную находили ту самую последовательность, на которой модель ошибалась.

> То есть, в ситуации, когда ваша задача является закрытой (у нее есть итоговое полное решение и конечный ответ),, набор данных маленьким (все пространство 243 варианта, в обучении 2/3 для теста 1/3) и по условию задачи вы получаете результат в 95 или 97 баллов, для данного примера, что соответствует 2 или 1 ошибочным предсказаниям - вы можете попытаться посмортеть, в каких предсказаниях модель неуверена более всего и использовать данную информацию для трансформации вашего ответа. Что действительно дает результат. Замечу, что не считаю этот способ чем-то глупым или плохим, по той причине, что вы диагностируете проблему и выявляете потенциальные аномалии сами, это работа с данными и это тоже надо уметь, правда обычно работа с данными это этап до обучения моделей, но вот перед нами оказался случай, когда нужно работать после, интересное напоминание о том, что понимание того, с чем и для чего вы работаете - важно.

Пропишу, что, если у вас была проблема с 95 и 97 баллами, вероятно, ваши нервы попортили последовательности WFWFW и FFWFW. Теперь попробуйте используя код выше объяснить себе - как именно они были выявлены.

In [17]:
evaluate_with_reference(submission_ens, REFERENCE_PATH)


=== Автопроверка по эталону ===
Попаданий: 71/73
Accuracy : 0.972603

Ошибочные руны:
 rune  spell_pred  spell_true
WFWFW           0           1
FFWFW           0           1


## Подход 2. SVM с подбором гиперпараметров

Второй вариант - использовать метод опорных векторов и подобрать его параметры с помощью `GridSearchCV`.

На практике именно этот подход оказался особенно сильным для данной задачи.

Почему это может происходить:

- признаков немного;
- структура признаков аккуратная и компактная;
- задача напоминает классификацию в хорошо определённом признаковом пространстве;
- SVM хорошо работает на небольших и средних датасетах с чёткой границей разделения.

Здесь мы перебираем 20 кандидатов:
- 4 варианта для линейного ядра;
- 16 вариантов для RBF-ядра.

In [10]:
start_total = time.time()

svm_search = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        random_state=42,
        probability=False,
        cache_size=1000
    ))
])

param_grid = [
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.1, 1, 10, 100]
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", 0.01, 0.1, 1]
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=svm_search,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

In [11]:
start_search = time.time()
grid.fit(X_train, y)
end_search = time.time()

print("\n=== Результат GridSearch ===")
print("Best params :", grid.best_params_)
print("Best CV acc :", grid.best_score_)
print(f"Время поиска: {end_search - start_search:.2f} сек")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

=== Результат GridSearch ===
Best params : {'svc__C': 0.1, 'svc__kernel': 'linear'}
Best CV acc : 0.9941176470588236
Время поиска: 0.65 сек


## Почему поиск делается без `probability=True`

Во время перебора параметров важно не тратить лишнее время.

Опция `probability=True` у `SVC` делает обучение заметно тяжелее, потому что дополнительно оцениваются вероятности.

Поэтому в `GridSearchCV` удобнее сначала искать лучшие параметры без вероятностей, а уже после этого обучить финальную модель с теми же параметрами, но с `probability=True`.

In [12]:
best_params = grid.best_params_
best_kernel = best_params["svc__kernel"]
best_c = best_params["svc__C"]
best_gamma = best_params.get("svc__gamma", "scale")

svm_final = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        kernel=best_kernel,
        C=best_c,
        gamma=best_gamma,
        probability=True,
        random_state=42,
        cache_size=1000
    ))
])

In [13]:
start_final = time.time()
svm_final.fit(X_train, y)
end_final = time.time()

print(f"Время финального fit: {end_final - start_final:.2f} сек")

Время финального fit: 0.02 сек


In [14]:
test_proba = svm_final.predict_proba(X_test)[:, 1]
test_pred_svm = (test_proba >= 0.5).astype(int)

submission_svm = pd.DataFrame({
    "rune": test_df["rune"],
    "spell": test_pred_svm
})

submission_svm.to_csv(OUTPUT_PATH, index=False)

print(f"Файл сохранён: {OUTPUT_PATH}")
submission_svm.head(10)

Файл сохранён: answers_you.csv


,rune,spell
0,FWFEF,1
1,EFEEE,0
2,EFWFE,0
3,FEWWF,0
4,WFEFE,0
5,EWFEW,0
6,WWFFW,1
7,FEWFW,0
8,WEFEW,1
9,EFFFF,0


In [15]:
debug_df = pd.DataFrame({
    "rune": test_df["rune"],
    "proba_1": test_proba,
    "pred": test_pred_svm,
    "uncertainty": np.abs(test_proba - 0.5)
}).sort_values("uncertainty", ascending=True)

print("Топ-10 самых неуверенных предсказаний SVM:")
print(debug_df.head(10).to_string(index=False))

Топ-10 самых неуверенных предсказаний SVM:
 rune  proba_1  pred  uncertainty
EWWWW 0.597106     1     0.097106
FFWFW 0.785418     1     0.285418
WFWFW 0.787462     1     0.287462
EFWWE 0.181139     0     0.318861
EWWWF 0.849200     1     0.349200
FFWWF 0.927469     1     0.427469
WFWWF 0.928362     1     0.428362
EEEWE 0.040571     0     0.459429
WFEFE 0.035323     0     0.464677
EWEEE 0.035085     0     0.464915


In [16]:
evaluate_with_reference(submission_svm, REFERENCE_PATH)
print(f"\nОбщее время скрипта: {time.time() - start_total:.2f} сек")


=== Автопроверка по эталону ===
Попаданий: 73/73
Accuracy : 1.000000
Ошибок нет.

Общее время скрипта: 20.18 сек


## Сравнение двух подходов

### Ансамбль
Плюсы:
- сильный baseline;
- позволяет анализировать неуверенные и спорные объекты;
- хорошо показывает, как разные модели видят задачу.

Минусы:
- может не добить до 100 баллов автоматически;
- чувствителен к тонким закономерностям, которые не всегда легко уловить деревьями.

### SVM
Плюсы:
- для этой задачи может давать идеальный результат;
- хорошо работает на компактных и информативных признаках;
- подбор параметров оказывается достаточно недорогим.

Минусы:
- требует более аккуратной настройки;
- нужно понимать влияние ядра, `C` и `gamma`.

## Итоговый вывод

Эта задача интересна тем, что нарушает популярный стереотип:

> более современные и популярные ансамбли деревьев не всегда лучше классических методов.

На структурированных коротких строках с хорошими признаками SVM способен показывать отличный результат и даже выигрывать у более “модных” подходов.

При этом ансамбль тоже остаётся полезным:

- как сильный baseline;
- как инструмент для анализа неуверенных объектов;
- как способ понять, где именно задача остаётся сложной.

Именно поэтому для учебного разбора полезно показывать оба подхода, а не только финальный победивший вариант.